In [10]:
from dotenv import load_dotenv
import openai
import os 
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('env')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
LANGSMITH_API_KEY=os.getenv('LANGSMITH_API_KEY')

os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['LANGCHAIN_PROJECT'] = 'LANG-1'

checked

## DataLoader

데이터 로더는 원본 자료를 읽어 Document(텍스트+메타데이터)로 바꿔주는 역할

자료의 포맷과 형식, 원리에 따라 다양한 로더가 존재

**대표적인 데이터로더**
- pyMuPDFLoader: PDF 정확·빠름(표를 읽을 수 있음)
- WebBaseLoader: URL에서 HTML 읽기(BeautifulSoup 사용). 정적 페이지에 적합.
- CSVLoader: CSV를 행(row) 단위 Document로 로드
- DirectoryLoader: 폴더 전체를 한 번에, loader_cls 로 하위 로더 지정.

In [3]:
%pip install -qU unstructured pdfminer pymupdf chromadb
%pip install -qU pdfminer.six==20221105

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ypy-websocket 0.8.4 requires aiofiles<23,>=22.1.0, but you have aiofiles 24.1.0 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.32.1 which is incompatible.
opentelemetry-exporter-prometheus 0.56b0 requires opentelemetry-sdk~=1.35.0, but you have opentelemetry-sdk 1.37.0 which is incompatible.
mlflow-skinny 2.21.3 requires packaging<25, but you have packaging 25.0 which is incompatible.
mlflow-skinny 2.21.3 requires protobuf<6,>=3.12.0, but you have protobuf 6.32.1 which is incompatible.
azureml-training-tabular 1.60.0 requires numpy<=1.23.5,>=1.16.0; python_version >= "3.8", but you have numpy 2.2.6 which is inco

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("./pdf/인터넷서비스_이용약관_202509.pdf", mode="page") #<-- mode="page"로 하면 페이지 단위로 나눠서 로드. 하나의 페이지가 하나의 document가 됨
docs = loader.load()
print(f"문서 개수: {len(docs)}")
print('---------------------------')
print(docs[1].page_content[:500]) #<-- 첫 페이지의 앞 500자 출력
print('---------------------------')
print(docs[1].metadata) #<-- 메타데이터

문서 개수: 232
---------------------------
-  2 - 
제1장  총   칙 
 
제 1조 ( 약관의 적용 ) 
① 주식회사 케이티(이하 "케이티"라 합니다)의 인터넷서비스(이하 "서비스"라 합니다)를 
이용할 때는 이 약관과 전기통신서비스이용기본약관(이하 “기본약관”이라 합니다)을 
함께 적용합니다. 
② 케이티인터넷서비스 이용약관은 지정된 홈페이지(www.kt.com)에 게시하여 공지합니다. 
 
제 2조 ( 정의 ) 
기본약관에서 정의한 용어 이외에 이 약관에서 사용하는 용어의 정의는 다음과 같습니다. 
1. 
KT internet: 케이티가 제공하는 초고속인터넷서비스 
2. 
KT biz kornet: 케이티가 구축한 인터넷망의 이름 또는 인터넷 전용회선서비스 
3. 
KT WiFi: 노트북, PDA, TABLET 등의 단말을 이용하여 가정, 기업, KT WiFi ZONE 
등에서 무선으로 초고속인터넷을 이용할 수 있는 서비스 
4. 
지니 TV VOD: KT internet 회선종단에 지니 TV 단말을 설치하고, 가
---------------------------
{'producer': 'Microsoft® Word Microsoft 365용\udcc0\udc80', 'creator': 'Microsoft® Word Microsoft 365용\udcc0\udc80', 'creationdate': '2025-09-03T15:58:56+09:00', 'source': './pdf/인터넷서비스_이용약관_202509.pdf', 'file_path': './pdf/인터넷서비스_이용약관_202509.pdf', 'total_pages': 232, 'format': 'PDF 1.7', 'title': '', 'author': '박현수(상품시너지팀)\udcc0\udc80', 'subject': '', 'keywords': '', 'moddate': '2025-09-03T15:58:56+09:00', 'trapped': '', 'modDat

### Splitter

읽어들인 데이터는 페이지 단위이므로 너무 많은 정보를 담고 있다.

이를 적절한 의미 단위로 나누는 과정이 필요하다.  

기본적으로 아래 예시처럼 chunk단위로 자르지만 이럴경우 문장이 잘려나갈 수 있으므로 overlap을 두어 안정성을 높인다.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", "  ", " "]
)
splits = splitter.split_documents(docs)
for s in splits[:5]:
    print(s.page_content)
    print('-----------------')

-  1 - 
 
 
 
KT 인터넷 이용약관 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
2025. 9
-----------------
-  2 - 
제1장  총   칙 
 
제 1조 ( 약관의 적용 ) 
① 주식회사 케이티(이하 "케이티"라 합니다)의 인터넷서비스(이하 "서비스"라 합니다)를 
이용할 때는 이 약관과 전기통신서비스이용기본약관(이하 “기본약관”이라 합니다)을 
함께 적용합니다. 
② 케이티인터넷서비스 이용약관은 지정된 홈페이지(www.kt.com)에 게시하여 공지합니다. 
 
제 2조 ( 정의 ) 
기본약관에서 정의한 용어 이외에 이 약관에서 사용하는 용어의 정의는 다음과 같습니다. 
1.
-----------------
1. 
KT internet: 케이티가 제공하는 초고속인터넷서비스 
2. 
KT biz kornet: 케이티가 구축한 인터넷망의 이름 또는 인터넷 전용회선서비스 
3. 
KT WiFi: 노트북, PDA, TABLET 등의 단말을 이용하여 가정, 기업, KT WiFi ZONE 
등에서 무선으로 초고속인터넷을 이용할 수 있는 서비스 
4. 
지니 TV VOD: KT internet 회선종단에 지니 TV 단말을 설치하고, 가정내 정보단말 
및 각종기기를 유무선으로 연결하여 고품질 VOD 및 홈네트워킹 등 다양한
-----------------
및 각종기기를 유무선으로 연결하여 고품질 VOD 및 홈네트워킹 등 다양한 
서비스를 이용할 수 있는 서비스 
5. 
고객 ID: 고객을 식별하고 서비스를 이용하게 하기 위하여 고객이 선정하고 
케이티가 부여하는 문자와 숫자 등의 조합  
6. 
비밀번호: 고객의 서비스 이용권을 보호하기 위하여 고객이 선정하는 문자와 숫자 
등의 조합 
7. 
IP주소(Internet Protocol address): 인터넷 망에서 송신원과 송신선을 식별하기 위해 
배정하는 주소 
8. 
IP단말기기: IP번호를 지니는 단말기기 
9.
-----------------
배정하는 주소 
8. 
IP

### Vectorization

잘려진 데이터는 의미를 간직한 채로 vector로 변경된다. 키워드 중심의 BM25가 있고 아래 예시는 의미 중심의 딥러닝 임베딩 모델을 사용했다.

벡터화 할 때마다 비용과 시간이 들어가므로 변경한 자료는 DB에 저장하는 것이 효율적이다.

In [13]:
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import Chroma, FAISS

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

emb = AzureOpenAIEmbeddings(
    model=EMB_MODEL_NAME,                      
    api_key=AZURE_OPENAI_EMB_API_KEY,
    azure_endpoint=EMB_END_POINT,
    api_version="2024-08-01-preview"
)
vectordb = FAISS.from_documents(splits,emb)
vectordb.save_local("faiss_index")# local에 저장

In [14]:
#로컬에 저장된 db를 불러올 때
vectordb = FAISS.load_local("faiss_index", emb, allow_dangerous_deserialization=True)

### Retriever

벡터 DB는 보통 리트리버를 포함하고 있다.

리트리버는 관련있는 정보를 검색하여 return한다.

**kargs**
- search_type : 관련 문서를 고르는 알고리즘
    - similarity : 유사한 문서 추출
    - similarity_score_threshold : 임계값 이상인 것들만 추출해서 관련 없는 정보를 배제
    - mmr : 너무 유사한 것들만 있으면 중복이 심하다. 다양한 정보를 가져와 편향을 방지

- k : 상위 k개 최종 선별



In [15]:
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 10})

In [16]:
emb_text = emb.embed_query("안녕하세요")
print(emb_text[:5])
print(len(emb_text)) #text-embedding-3-small 모델은 어떤 문장이 들어가도 1536차원 벡터를 리턴

[-0.002531265141442418, -0.06127675995230675, -0.008443817496299744, 0.031540773808956146, 0.031089577823877335]
1536


meta data에는 원본 문서의 다양한 정보가 포함되어 있다. 

대표적으로 파일의 이름, page 등이 있다. 메타 정보는 자동으로 뽑히기도 하지만 수동으로 추가할 수도 있다.

In [19]:
for doc in retriever.get_relevant_documents("가정의 달에 가입하면 혜택이 뭐야?")[:5]:  
    print(doc.metadata['page'], doc.metadata['source'], doc.metadata['creationdate'], doc.metadata['author'])


222 ./pdf/인터넷서비스_이용약관_202509.pdf 2025-09-03T15:58:56+09:00 박현수(상품시너지팀)��
130 ./pdf/인터넷서비스_이용약관_202509.pdf 2025-09-03T15:58:56+09:00 박현수(상품시너지팀)��
223 ./pdf/인터넷서비스_이용약관_202509.pdf 2025-09-03T15:58:56+09:00 박현수(상품시너지팀)��
64 ./pdf/인터넷서비스_이용약관_202509.pdf 2025-09-03T15:58:56+09:00 박현수(상품시너지팀)��
93 ./pdf/인터넷서비스_이용약관_202509.pdf 2025-09-03T15:58:56+09:00 박현수(상품시너지팀)��


찾아온 정보가 너무 많거나 혹은 정보의 밀도가 낮은 경우 context를 바로 프롬프트에 넣기 전에 요약을 할 수 있다.

In [20]:
# (Optional)
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,  
    azure_deployment=MODEL_NAME,          
    api_version="2024-12-01-preview",
    temperature=0.2,
)

compressor = LLMChainExtractor.from_llm(llm)
compressed_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever # compressor를 설정하면 압축된 context를 가져옴
)

## RAG 만들기

기본적으로 일반챗봇과 동일하지만 context를 넣는 위치가 존재하고 프롬프트에 context를 강하게 참조라하고 명령

In [21]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 문서 기반 QA 어시스턴트다. 아래 규칙을 지켜라.\n"
     "1) 제공된 컨텍스트(표/본문)에서만 근거를 찾아 답한다.\n"
     "2) 모르면 모른다고 말한다. 추측 금지.\n"
     "3) 숫자/조건/예외는 정확히 인용하고, 출처 문장 일부를 함께 제시한다.\n"
     "4) 표 내용이라면 행/열 헤더를 함께 언급해 맥락을 명확히 한다.\n"
     "5) 출처 페이지를 함께 제시한다.\n"),
    ("human", "질문: {question}\n\n컨텍스트:\n{context}\n\n한국어로 간결하게 답해.")
])


체인을 만들어 실행해봅시다.

RunnablePassthrough는 입력 변수를 바로 통과시키는 역할을 합니다.

In [22]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

parser = StrOutputParser()

def format_docs(docs):
    # 표 행/열 헤더 유지를 위해 원문을 크게 변형하지 않고 이어붙임
    return "\n\n---\n\n".join(d.page_content for d in docs)

# RAG 체인: (질문) → 검색 → 컨텍스트 → 프롬프트에 추가 → LLM → 텍스트
rag_chain = (
    {"question": RunnablePassthrough(),   # 입력 쿼리는 question과 retriever로 나뉘어 들어간다. 
     "context": retriever | format_docs}   # retriever를  compressed_retriever로도 변경해 봅시다
    | prompt
    | llm
    | parser
)


주어진 쿼리를 토대로 가장 관련된 내용을 찾아 답변하는 것을 확인할 수 있습니다

In [25]:
query = "오피스IP팩은 어떤 서비스야?"
print(rag_chain.invoke(query))

오피스IP팩은 인터넷을 이용하는 고객 대상으로 제공하는 고정IP 또는 유동 IP 제공 서비스입니다. 신규 가입 고객에게 3년 약정 시 가입설치비 면제와 추가 단말 30대 제공 혜택이 있으며, 오피스넷 기본상품 기반으로 회선 당 최대 3개까지 고정IP를 제공합니다. (출처: 94쪽, 127쪽)


## RAG 통합코드

In [ ]:
# pip install --upgrade "chromadb==0.4.24" "langchain-community>=0.3.0" pymupdf -q

In [29]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("pdf/인터넷서비스_이용약관_202509.pdf", mode="page") #<-- mode="page"로 하면 페이지 단위로 나눠서 로드. 하나의 페이지가 하나의 document가 됨
docs = loader.load()

print('데이터 로드 완료')

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n\n", "\n", "  ", " "]
)
splits = splitter.split_documents(docs)

print('분할 완료')

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

emb = AzureOpenAIEmbeddings(
    model=EMB_MODEL_NAME,                      
    api_key=AZURE_OPENAI_EMB_API_KEY,
    azure_endpoint=EMB_END_POINT,
    api_version="2024-08-01-preview"
)
vectordb = FAISS.from_documents(splits,emb)
print('임베딩 및 저장 완료')


데이터 로드 완료
분할 완료
임베딩 및 저장 완료


In [30]:
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 10})

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 문서 기반 QA 어시스턴트다. 아래 규칙을 지켜라.\n"
     "1) 제공된 컨텍스트(표/본문)에서만 근거를 찾아 답한다.\n"
     "2) 모르면 모른다고 말한다. 추측 금지.\n"
     "3) 숫자/조건/예외는 정확히 인용하고, 출처 문장 일부를 함께 제시한다.\n"
     "4) 표 내용이라면 행/열 헤더를 함께 언급해 맥락을 명확히 한다.\n"
     "5) 출처 페이지를 함께 제시한다.\n"),
    ("human", "질문: {question}\n\n컨텍스트:\n{context}\n\n한국어로 간결하게 답해.")
])

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,  
    azure_deployment=MODEL_NAME,          
    api_version="2024-12-01-preview",
    temperature=0.2,
)
parser = StrOutputParser()

def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"question": RunnablePassthrough(),
     "context": retriever | format_docs}   
    | prompt
    | llm
    | parser
)

query = "KT 사장님 배달POS의 비용은?"
print(rag_chain.invoke(query))

KT 사장님 배달POS의 비용은 월 22,000원입니다. 서비스 가입 시 최초 1회 2개월 간 월정액 요금 전액 할인 혜택이 있으며, 출시 기념 프로모션으로 기간 내 가입 시 월정액 요금 할인 혜택 1개월 연장이 적용됩니다(기간: 2022.9.01 ~ 2023.2.28).  
출처: 94쪽 "KT 사장님 배달POS" 항목.
